# Godot 生态仓库 · 颜色矩阵

数据来自本仓库 `godot.db`(5 个 topic、`star ≥ 1000`、最近 3 个月有推送,去重后 51 个仓库)。

矩阵维度:**full_name / url / stars / forks / open_issues / 月均新增 star**。

> 月均新增 star = `stars ÷ 从创建到最后更新的月数`,衡量仓库历史上的平均涨星速度。
数值列按列做热力着色(颜色越深 = 数值越高)。

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('godot.db')
df = pd.read_sql_query(
    'SELECT full_name, url, stars, forks, open_issues, created_at, updated_at FROM repos',
    conn,
)
conn.close()
len(df)

51

In [2]:
# 月均新增 star = stars / 从创建到最后更新的月数(至少按 1 个月计,避免除零)
created = pd.to_datetime(df['created_at'], utc=True)
updated = pd.to_datetime(df['updated_at'], utc=True)
active_months = ((updated - created).dt.days / 30.44).clip(lower=1.0)
df['stars_per_month'] = (df['stars'] / active_months).round(1)
df = df.drop(columns=['created_at', 'updated_at'])
df = df.sort_values('stars', ascending=False).reset_index(drop=True)
df.head()

,full_name,url,stars,forks,open_issues,stars_per_month
0,godotengine/godot,https://github.com/godotengine/godot,112899,25726,18480,755.1
1,Donchitos/Claude-Code-Game-Studios,https://github.com/Donchitos/Claude-Code-Game-...,22010,3181,44,5153.7
2,heroiclabs/nakama,https://github.com/heroiclabs/nakama,12778,1430,124,112.9
3,godotengine/awesome-godot,https://github.com/godotengine/awesome-godot,10210,549,53,76.0
4,Orama-Interactive/Pixelorama,https://github.com/Orama-Interactive/Pixelorama,9771,516,83,119.0


In [3]:
# 颜色矩阵:数值列逐列热力着色;url 渲染为可点击链接
num_cols = ['stars', 'forks', 'open_issues', 'stars_per_month']

styled = (
    df.style
      .background_gradient(cmap='YlGnBu', subset=['stars'])
      .background_gradient(cmap='OrRd', subset=['forks'])
      .background_gradient(cmap='Purples', subset=['open_issues'])
      .background_gradient(cmap='Greens', subset=['stars_per_month'])
      .format({'stars': '{:,}', 'forks': '{:,}', 'open_issues': '{:,}',
               'stars_per_month': '{:.1f}'})
      .format({'url': lambda u: f'<a href="{u}" target="_blank">link</a>'})
      .set_caption('Godot 生态仓库颜色矩阵 — 按 stars 降序')
      .set_properties(subset=['full_name'], **{'text-align': 'left',
                                               'font-weight': 'bold'})
      .set_table_styles([
          {'selector': 'caption',
           'props': [('font-size', '16px'), ('font-weight', 'bold'),
                     ('padding', '8px')]},
          {'selector': 'th',
           'props': [('background-color', '#222'), ('color', 'white'),
                     ('padding', '6px 10px')]},
          {'selector': 'td', 'props': [('padding', '4px 10px')]},
      ])
)
styled

,full_name,url,stars,forks,open_issues,stars_per_month
0,godotengine/godot,link,112899,25726,18480,755.100000
1,Donchitos/Claude-Code-Game-Studios,link,22010,3181,44,5153.700000
2,heroiclabs/nakama,link,12778,1430,124,112.900000
3,godotengine/awesome-godot,link,10210,549,53,76.000000
4,Orama-Interactive/Pixelorama,link,9771,516,83,119.000000
5,0xFA11/MultiplayerNetworkingResources,link,8556,534,2,84.800000
6,Redot-Engine/redot-engine,link,5880,306,61,56.400000
7,dialogic-godot/dialogic,link,5726,335,168,79.000000
8,RodZill4/material-maker,link,5554,352,311,58.500000
9,godotengine/godot-docs,link,5430,3686,1077,43.000000


## 维度速览

- **stars / forks / open_issues**:GitHub 即时指标,采集时快照。
- **stars_per_month(月均新增 star)**:用 `stars` 除以「创建→最后更新」跨度的月数,近似该仓库生命周期内的平均涨星速度。注意这是历史平均,不等于当前热度——老牌项目即便现在仍火,均值也会被早期低速期摊薄;新项目则可能偏高。

In [4]:
# 月均涨星 Top 15(换个角度看“增长效率”而非绝对体量)
top_growth = df.sort_values('stars_per_month', ascending=False).head(15).reset_index(drop=True)
(top_growth[['full_name', 'stars', 'stars_per_month']]
    .style
    .background_gradient(cmap='Greens', subset=['stars_per_month'])
    .format({'stars': '{:,}', 'stars_per_month': '{:.1f}'})
    .set_caption('月均新增 star Top 15'))

,full_name,stars,stars_per_month
0,Donchitos/Claude-Code-Game-Studios,"22,010",5153.7
1,htdt/godogen,"3,583",807.9
2,godotengine/godot,"112,899",755.1
3,MattParkerDev/SharpIDE,"3,774",510.6
4,KsanaDock/Microverse,"2,375",283.5
5,Coding-Solo/godot-mcp,"4,319",273.9
6,Orama-Interactive/Pixelorama,"9,771",119.0
7,heroiclabs/nakama,"12,778",112.9
8,godot-rust/gdext,"4,904",103.2
9,TokisanGames/Terrain3D,"3,988",97.4
